这个文本用于找出美国与中国的股票市场的共同交易日,并且将这些数据集进行对齐,保留下来用于未来的训练.

In [1]:
from utils.common import *

In [2]:
all_index_dic = pd.read_pickle('src/pickle/all_index_dic.pkl')
all_index_dic.keys()

Index(['CSI_300', 'A_500', 'metal', 'equal', 'weight', 'cap', 'gold'], dtype='object')

In [3]:
CSI_300 = all_index_dic.CSI_300
A_500 = all_index_dic.A_500
metal = all_index_dic.metal
equal = all_index_dic.equal
weight = all_index_dic.weight
cap = all_index_dic.cap
gold = all_index_dic.gold

In [4]:
CSI_300

2012-05-09    1000.000000
2012-05-10     999.887113
2012-05-11     992.249523
2012-05-14     984.203282
2012-05-15     984.895658
                 ...     
2026-07-20    1730.309229
2026-07-21    1783.331715
2026-07-22    1775.058946
2026-07-23    1779.107090
2026-07-24    1749.451555
Name: CSI_300, Length: 3690, dtype: float64

In [5]:
A_500

2012-05-09    1000.000000
2012-05-10    1000.166229
2012-05-11     993.645535
2012-05-14     986.340876
2012-05-15     986.907436
                 ...     
2026-07-20    2040.782832
2026-07-21    2119.573044
2026-07-22    2106.458340
2026-07-23    2110.372354
2026-07-24    2070.198546
Name: A_500, Length: 3690, dtype: float64

In [6]:
metal

2012-05-09    1000.000000
2012-05-10    1011.510083
2012-05-11     998.374886
2012-05-14     978.325328
2012-05-15     973.850869
                 ...     
2026-07-20    1408.661095
2026-07-21    1471.372538
2026-07-22    1519.607747
2026-07-23    1568.733354
2026-07-24    1492.985508
Name: metal, Length: 3690, dtype: float64

In [7]:
equal

2012-05-09    1000.000000
2012-05-10    1015.423398
2012-05-11    1016.440364
2012-05-14    1016.659495
2012-05-15    1016.133666
                 ...     
2026-07-20    2310.742688
2026-07-21    2456.527008
2026-07-22    2615.889369
2026-07-23    2650.264085
2026-07-24    2503.439394
Name: equal, Length: 3690, dtype: float64

In [8]:
weight

2012-05-09    1000.000000
2012-05-10    1001.570724
2012-05-11     992.267054
2012-05-14     982.831739
2012-05-15     982.433532
                 ...     
2026-07-20    5183.388003
2026-07-21    5393.500787
2026-07-22    5749.830273
2026-07-23    5878.439922
2026-07-24    5595.106954
Name: weight, Length: 3690, dtype: float64

In [9]:
cap

2012-05-09    1000.000000
2012-05-10    1002.567089
2012-05-11     993.729282
2012-05-14     984.653691
2012-05-15     984.558675
                 ...     
2026-07-20    4872.405733
2026-07-21    5081.104789
2026-07-22    5417.074316
2026-07-23    5532.429089
2026-07-24    5262.212255
Name: cap, Length: 3690, dtype: float64

In [10]:
gold

2012-05-09    1000.000000
2012-05-10    1000.878474
2012-05-11     993.662561
2012-05-14     979.230736
2012-05-15     976.846393
                 ...     
2026-07-20    2516.345719
2026-07-21    2554.495967
2026-07-22    2602.058122
2026-07-23    2539.122935
2026-07-24            NaN
Name: gold, Length: 3690, dtype: float64

经检查,他们的格式是一样的,datetime作为index,我们要找出他们的交集.

In [11]:
from functools import reduce
dfs = [CSI_300, A_500, metal, equal, weight, cap, gold]
common_dates = reduce(lambda x, y: x.intersection(y), [df.index for df in dfs])

In [12]:
common_dates

DatetimeIndex(['2012-05-09', '2012-05-10', '2012-05-11', '2012-05-14',
               '2012-05-15', '2012-05-16', '2012-05-17', '2012-05-18',
               '2012-05-21', '2012-05-22',
               ...
               '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16',
               '2026-07-17', '2026-07-20', '2026-07-21', '2026-07-22',
               '2026-07-23', '2026-07-24'],
              dtype='datetime64[ns]', length=3690, freq=None)

到这里意识到一个问题:为什么这些数据都是3690条?A股筛选过后的条数应该在3454天,黄金期货应该是3570天,理论上数据应该更小.  
   
   
--- 
原因是,因为我在整理 **字典** 的时候,每一个数据都是一个Series,如上所示,datetime作为index,数据只有一个归一化后的百分比变化,因此数据形式是Series,在字典中会取并集,因此会出现数据变多的现象.

In [13]:
CSI_300.isna().sum()

np.int64(236)

236+3454 = 3690 发现原因了,因为取并集的时候对应天数并没有数据,所以出现了这个情况.因此,我们应该先把na给去除掉,再取共同的天数.

In [14]:
CSI_300.dropna(inplace=True)
A_500.dropna(inplace=True)
equal.dropna(inplace=True)
weight.dropna(inplace=True)
metal.dropna(inplace=True)
cap.dropna(inplace=True)
gold.dropna(inplace=True)

In [15]:
CSI_300.describe()

count    3454.000000
mean     1386.531293
std       308.682470
min       785.307998
25%      1218.348991
50%      1420.107665
75%      1552.097844
max      2185.395449
Name: CSI_300, dtype: float64

In [16]:
dfs = [CSI_300, A_500, metal, equal, weight, cap, gold]
common_dates = reduce(lambda x, y: x.intersection(y), [df.index for df in dfs])
display(common_dates)

DatetimeIndex(['2012-05-09', '2012-05-10', '2012-05-11', '2012-05-14',
               '2012-05-15', '2012-05-16', '2012-05-17', '2012-05-18',
               '2012-05-21', '2012-05-22',
               ...
               '2026-07-10', '2026-07-13', '2026-07-14', '2026-07-15',
               '2026-07-16', '2026-07-17', '2026-07-20', '2026-07-21',
               '2026-07-22', '2026-07-23'],
              dtype='datetime64[ns]', length=3334, freq=None)

这回数据变成了3334,数据应该是对的,根据此来重新筛选数据,保存起来.

In [17]:
CSI_300 = CSI_300.loc[common_dates]
A_500 = A_500.loc[common_dates]
equal = equal.loc[common_dates]
weight = weight.loc[common_dates]
metal = metal.loc[common_dates]
cap = cap.loc[common_dates]
gold = gold.loc[common_dates]

In [20]:
all_index_dic =pd.DataFrame({
    'CSI_300': CSI_300,
    'A_500': A_500,
    'metal': metal,
    'equal': equal,
    'weight': weight,
    'cap': cap,
    'gold': gold
})

all_index_dic.to_pickle('src/pickle/all_index_dic.pkl')

In [21]:
all_index_dic.describe()

,CSI_300,A_500,metal,equal,weight,cap,gold
count,3334.000000,3334.000000,3334.000000,3334.000000,3334.000000,3334.000000,3334.000000
mean,1386.858780,1540.535532,889.563787,1073.978108,1712.994252,1660.412068,1138.609546
std,308.756455,379.047547,273.683205,547.405621,1377.647102,1293.478711,520.858201
min,785.307998,794.663091,533.563259,502.177783,510.499173,512.583926,659.346226
25%,1218.348991,1339.316565,704.520468,774.398949,833.305628,840.385384,800.291810
50%,1420.308792,1556.028938,841.118802,937.075504,982.308470,987.768160,996.737183
75%,1551.974138,1759.764437,980.216331,1150.824190,2185.860196,2091.356742,1204.477029
max,2185.395449,2387.145456,2239.823775,4338.775772,8301.665880,7946.140253,3337.140030
